# Görev G — TBPTT + decay otopsisi: §20a hipotezinin belirleyici testi

**Hipotez (§20a):** Öğrenilmiş decay ~0.949/token → state tek chunk'ta (256 token)
~2·10⁻⁶'ya siliniyor; ve sınırda detach olduğu için model daha yavaş unutmayı
**öğrenemiyor** (cross-chunk hata decay'e hiç gradyan taşımıyor).

**Müdahale:** `CC_BPTT=1` — chunk sınırında detach kaldırılır
(`bptt_across_chunks=True`, K2/TBPTT yolu); B'nin cross-chunk hatası decay/yazma
parametrelerine geri akar. Eğitim protokolünün geri kalanı Görev E ile birebir aynı.

**ÖN-KAYITLI KRİTERLER:**
- **HİPOTEZ DOĞRULANDI:** TBPTT kollarında (a) λ_max ≥ 0.995 kanallar oluşur
  (256-token sağkalım ≥ %28) VE (b) uzak gap'lerde (4096/16384) seed-ort ≥ %15.
  → §17-§20 zinciri "decay kalibrasyonu + detach" eğitim-reçetesi bulgusuna indirgenir.
- **KISMİ:** λ büyüyor ama doğruluk gelmiyor → okuma yolu sıradaki şüpheli.
- **RED:** λ büyümüyor → optimizasyon sinyali yetersiz; farklı LR/adım denenir.
- Karşılaştırma tabanı §19 (detach'li, aynı protokol): uzak gap'ler ≤ %4.4;
  §17 otopsisi decay ort 0.949.

GPU **T4**, Internet **On** → Run All (~60-80 dk; TBPTT geri yayılımı daha ağır).

In [ ]:
# --- 1. Repo ---
import subprocess, sys, os, re
os.chdir('/kaggle/working')
if not os.path.isdir('HFP'):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git'], check=True)
else:
    subprocess.run(['git','-C','HFP','pull'], check=True)
os.chdir('/kaggle/working/HFP')
import torch
assert torch.cuda.is_available(), 'GPU secili degil (T4)'
print('GPU:', torch.cuda.get_device_name(0))
s = open('review_scripts/carry_curriculum.py').read()
assert 'CC_BPTT' in s, 'script eski -> once bilgisayarda `git push` yapin!'
ENV = dict(os.environ, PYTHONPATH='/kaggle/working/HFP')
print('hazir (CC_BPTT destekli).')

In [ ]:
# --- 2. TBPTT KOLLARI: 2 mod x 3 seed (egitim + script-ici lifetime eval) ---
raw = {}
for m in ['exp', 'cubic_flux_chunked']:
    for s in ['0', '1', '2']:
        print(f'\n===== TBPTT | {m} | seed {s} =====', flush=True)
        r = subprocess.run([sys.executable, 'review_scripts/carry_curriculum.py', m, s, '2400'],
                           env=dict(ENV, CC_BPTT='1', CC_CARRY_MAX='16', CC_STEPS='1200'),
                           check=True, capture_output=True, text=True)
        for line in r.stdout.strip().splitlines():
            if 'step ' in line or 'gap ' in line or 'dogrulama' in line or 'FINAL' in line or 'CKPT' in line:
                print(line, flush=True)
        if 'butce bitti' in r.stdout:
            print('>> butce doldu — ayni hucreyi TEKRAR calistirin (resume eder).', flush=True)
        mm = re.search(r"FINAL.*?: (\{.*\})", r.stdout)
        if mm: raw[(m, s)] = eval(mm.group(1))
print('\nham:', raw)

In [ ]:
# --- 3. DECAY OTOPSISI: TBPTT sonrasi lambda dagilimi ---
import torch, math, glob
print(f'{"kol":>32} | {"lam_ort":>7} {"lam_max":>7} {">0.99":>6} {">0.995":>6} | {"max 256-tok sagkalim":>20}')
print('-'*92)
autopsy = {}
for f in sorted(glob.glob('checkpoints/carryv1t_*.pt')):
    st = torch.load(f, map_location='cpu', weights_only=False)
    sd = st['m'] if 'm' in st else st
    lam = torch.cat([torch.sigmoid(v.float().flatten()) for k, v in sd.items()
                     if 'decay' in k and 'bulk' in k.lower() or k.endswith('decay')])
    if lam.numel() == 0:
        lam = torch.cat([torch.sigmoid(v.float().flatten()) for k, v in sd.items() if 'decay' in k])
    mx = lam.max().item()
    autopsy[f] = mx
    print(f'{os.path.basename(f):>32} | {lam.mean():.4f} {mx:.4f} '
          f'{(lam>0.99).sum().item():>6} {(lam>0.995).sum().item():>6} | {mx**256*100:>19.1f}%')
print('\nReferans (§17, detach ile): lam ort 0.949 -> 256-tok sagkalim ~0.0002%')

In [ ]:
# --- 4. OZET + on-kayitli okuma ---
import statistics as st
GAPS = [256, 1024, 4096, 16384]
BASE19 = {'exp': {256:6.7,1024:5.5,4096:3.3,16384:1.1},
          'cubic_flux_chunked': {256:20.0,1024:4.4,4096:4.4,16384:2.2}}   # §19 detach'li
print(f'{"gap":>7} | {"exp §19":>8} {"exp TBPTT":>10} | {"cubic §19":>10} {"cubic TBPTT":>12}')
print('-'*58)
new = {}
for m in BASE19:
    for g in GAPS:
        vals = [raw[(m,s)][g][0] if isinstance(raw[(m,s)][g], tuple) else raw[(m,s)][g]
                for s in ['0','1','2'] if (m,s) in raw and g in raw[(m,s)]]
        new[(m,g)] = st.mean(vals) if vals else float('nan')
for g in GAPS:
    print(f'{g:>7} | {BASE19["exp"][g]:>7.1f}% {new[("exp",g)]:>9.1f}% | '
          f'{BASE19["cubic_flux_chunked"][g]:>9.1f}% {new[("cubic_flux_chunked",g)]:>11.1f}%')

far = [max(new[('exp',g)], new[('cubic_flux_chunked',g)]) for g in (4096, 16384)]
lam_ok = any(v >= 0.995 for v in autopsy.values())
print(f'\nUzak gap en iyi: {far[0]:.1f}% / {far[1]:.1f}%  |  lam>=0.995 kanal olustu mu: {lam_ok}')
if lam_ok and min(far) >= 15:
    print('OKUMA: HIPOTEZ DOGRULANDI -> §17-§20 zinciri "decay + detach" egitim-recetesi')
    print('  bulgusuna indirgendi. Mimari SAGLAM; TBPTT/decay-init receteye girer.')
elif lam_ok:
    print('OKUMA: KISMI -> decay yavaslamayi ogrendi ama dogruluk gelmedi; siradaki')
    print('  supheli OKUMA yolu (retrieval/norm).')
else:
    print('OKUMA: RED -> lambda buyumedi; optimizasyon sinyali yetersiz (LR/adim/mufredat')
    print('  ayari) ya da decay parametrizasyonu sinira izin vermiyor — incelenecek.')
print('\nCiktiyi Claude\'a yapistirin; RESULTS §21 olarak islenecek.')